# GA4 Flatten Events — BigQuery Spark Notebook

Self-contained BigQuery Studio PySpark notebook derived from `pipelines/spark/ga4_flatten_events.py`.

It flattens the nested GA4 `event_params` (and, for ecommerce events, `items`) into a wide, pivoted structure and writes the result back to BigQuery.

**Differences from the original Dataproc job:**

- **Input** is read **directly from BigQuery** via the Spark BigQuery connector (no GCS parquet load).
- All logic (previously split across `BaseSparkJob` + `SparkJob`) is inlined into **this single notebook**.
- **Output** is written **directly to BigQuery** using the `direct` write method (BigQuery Storage Write API — no temporary GCS bucket).

> Fill in the placeholder table names in the **Configuration** cell before running.

> **Note:** this notebook is kept as a self-contained BigQuery-Studio reference. The same logic now also ships as the installable [`ga4_sparkify`](./ga4_sparkify) package, which is storage- and service-agnostic (any filesystem / BigQuery, any Spark engine). In a managed-Spark notebook you can instead do:
>
> ```python
> from ga4_sparkify import FlattenEventsJob
> from ga4_sparkify.io.bigquery import BigQueryReader, BigQueryWriter
>
> FlattenEventsJob(
>     reader=BigQueryReader(table=SOURCE_TABLE),
>     writer=BigQueryWriter(DESTINATION_PROJECT, DESTINATION_DATASET, EVENT_NAME,
>                           write_mode="backfill", write_method="direct",
>                           partition_field="event_date", cluster_fields="user_pseudo_id"),
>     spark=spark, event_name=EVENT_NAME, event_type=EVENT_TYPE,
> ).run()
> ```

## 1. Imports & Spark session

In [ ]:
import logging

import pyspark.sql.functions as F
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.types import LongType
from pyspark.sql.window import Window

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

# In BigQuery Studio a Spark session is usually available; getOrCreate() is safe either way.
spark = SparkSession.builder.appName("ga4_flatten_events").getOrCreate()

## 2. Configuration

Equivalent to the CLI arguments of the original job. Replace the placeholder source/destination tables.

- `EVENT_NAME` — event to flatten (used as the destination table name).
- `EVENT_TYPE` — `"ecommerce"` triggers `items` explode/collect; anything else skips it.
- `EVENT_LIST` / `EVENT_PATTERN` — optional filters. Precedence: `EVENT_LIST` → `EVENT_PATTERN` → exact `EVENT_NAME`.
- `ADDITIONAL_COLUMNS_TO_SELECT` — extra top-level columns to carry through.
- `RUN_DATE` — logical date (`YYYY-MM-DD`); used to target the partition being (over)written.

In [ ]:
# --- Source (read directly from BigQuery) -------------------------------------
# Placeholder GA4 export table. For a single day use the dated shard, e.g.
#   your-project.analytics_XXXXXXXXX.events_20240101
# To read a range, use SOURCE_QUERY instead of SOURCE_TABLE (see the load cell).
SOURCE_TABLE = "your-project.analytics_XXXXXXXXX.events_20240101"
SOURCE_QUERY = None  # e.g. "SELECT * FROM `your-project.analytics_XXXXXXXXX.events_*` WHERE _TABLE_SUFFIX = '20240101'"

# --- Destination (write directly to BigQuery) ---------------------------------
DESTINATION_PROJECT = "your_project"
DESTINATION_DATASET = "dataset"

# --- Event selection ----------------------------------------------------------
EVENT_NAME = "purchase"
EVENT_TYPE = "ecommerce"  # "ecommerce" or "other"
EVENT_LIST = []            # e.g. ["purchase", "refund"]; takes precedence over EVENT_PATTERN
EVENT_PATTERN = None       # e.g. "view_%"; used only when EVENT_LIST is empty
ADDITIONAL_COLUMNS_TO_SELECT = []  # e.g. ["traffic_source"]

# --- Run metadata -------------------------------------------------------------
RUN_DATE = "2024-01-01"          # YYYY-MM-DD
RUN_DATE_NODASH = RUN_DATE.replace("-", "")

DESTINATION_TABLE = EVENT_NAME  # matches the original job's convention
PARTITION_FIELD = "event_date"
CLUSTER_FIELDS = "user_pseudo_id"

## 3. Processing logic

The `GA4FlattenEventsJob` class below inlines the transformation steps from the original `SparkJob`. Only the load/write have been re-pointed to BigQuery.

In [ ]:
class GA4FlattenEventsJob:
    """Flattens nested GA4 event data. All transformation logic is self-contained."""

    def __init__(self, spark: SparkSession):
        self.spark = spark

    def process_data(
        self,
        data: DataFrame,
        event_name: str,
        event_type: str,
        additional_columns_to_select: list,
        event_list: list = None,
        event_pattern: str = None,
    ) -> DataFrame:
        """Main processing pipeline."""
        logger.info("Starting data processing pipeline")

        data = self.repartition_and_filter(data, event_name, event_list, event_pattern)

        # DataFrame.isEmpty() instead of data.rdd.isEmpty() -- the RDD API is not
        # implemented on Spark Connect (BigQuery managed Spark).
        if data.isEmpty():
            logger.warning(
                f"No data found after filtering for event_name='{event_name}'. "
                "Skipping further processing."
            )
            return data

        data = self.create_default_columns(data)
        data = self.select_and_explode(data, additional_columns_to_select)
        data = self.pivot(data, additional_columns_to_select)
        data = self.create_additional_columns(data, additional_columns_to_select)
        if event_type == "ecommerce":
            data = self.explode_items(data)
            data = self.items_to_array(data)
        data = self.post_process(data)

        logger.info("Data processing completed")
        return data

    def repartition_and_filter(
        self,
        data: DataFrame,
        event_name: str,
        event_list: list = None,
        event_pattern: str = None,
    ) -> DataFrame:
        """Filters for events and repartitions to a fixed width."""
        if event_list:
            data = data.where(F.col("event_name").isin(*event_list))
        elif event_pattern:
            data = data.where(F.col("event_name").like(event_pattern))
        else:
            data = data.where(F.col("event_name") == event_name)

        # The original job inspected data.rdd.getNumPartitions() to choose between
        # repartition() and coalesce(). The RDD API is unavailable on Spark Connect,
        # so we repartition to a fixed 128 unconditionally (always yields 128
        # partitions; the coalesce branch was only a shuffle-avoidance optimization).
        return data.repartition(128)

    def create_default_columns(self, data: DataFrame) -> DataFrame:
        """Adds default columns if they are missing."""
        column_defaults = {
            "batch_event_index": F.lit(0).cast(LongType()),
            "batch_page_id": F.lit(0).cast(LongType()),
            "batch_ordering_id": F.lit(0).cast(LongType()),
        }
        for col_name, default_value in column_defaults.items():
            if col_name not in data.columns:
                data = data.withColumn(col_name, default_value)
                logger.debug(f"Added missing column: {col_name}")
        return data

    def select_and_explode(
        self, data: DataFrame, additional_columns_to_select: list
    ) -> DataFrame:
        """Selects relevant columns and explodes 'event_params'."""
        logger.info("Selecting and exploding 'event_params'")

        selected_columns = [
            "user_pseudo_id", "user_id", "event_name", "event_date", "event_timestamp",
            "device", "geo", "event_params", "event_previous_timestamp",
            "event_value_in_usd", "event_bundle_sequence_id",
            "event_server_timestamp_offset", "stream_id", "platform",
            "event_dimensions", "batch_event_index", "batch_page_id",
            "batch_ordering_id", "items", "user_properties",
        ]
        if additional_columns_to_select:
            selected_columns.extend(additional_columns_to_select)

        data = (
            data.select(
                *[F.col(col) for col in selected_columns],
                F.explode("event_params").alias("event_param"),
            )
            .withColumn("key", F.col("event_param.key"))
            .withColumn("string_value", F.col("event_param.value.string_value"))
            .withColumn("int_value", F.col("event_param.value.int_value"))
            .withColumn("float_value", F.col("event_param.value.float_value"))
            .withColumn("double_value", F.col("event_param.value.double_value"))
        )
        return data

    def pivot(self, data: DataFrame, additional_columns_to_select: list) -> DataFrame:
        """Pivots exploded data to key-value format."""
        logger.info("Pivoting data based on event parameters")
        pivot_keys = [
            row["key"]
            for row in data.where(F.col("key") != "items").select("key").distinct().collect()
        ]

        groupby_columns = [
            "user_pseudo_id", "user_id", "event_name", "event_date", "event_timestamp",
            "geo", "device", "items", "user_properties", "batch_event_index",
        ]
        if additional_columns_to_select:
            groupby_columns.extend(additional_columns_to_select)

        data = data.withColumn(
            "value",
            F.coalesce(
                F.col("string_value"),
                F.col("int_value").cast("string"),
                F.col("float_value").cast("string"),
                F.col("double_value").cast("string"),
            ),
        )
        data = (
            data.drop(
                "event_params", "event_param", "string_value", "int_value",
                "float_value", "double_value", "event_dimensions",
            )
            .where(F.col("key") != "items")
            .groupBy(groupby_columns)
            .pivot("key", pivot_keys)
            .agg(F.first("value"))
        )

        pivot_columns = [
            F.col(column)
            for column in data.columns
            if column
            not in (
                "key", "user_pseudo_id", "user_id", "event_name", "event_date",
                "event_timestamp", "geo", "device", "items", "user_properties",
                "batch_event_index", *additional_columns_to_select,
            )
        ]

        data = data.withColumn("event_params", F.struct(*pivot_columns))
        data = data.drop(*pivot_columns)
        return data

    def create_additional_columns(
        self, data: DataFrame, additional_columns_to_select: list
    ) -> DataFrame:
        """Creates additional computed columns."""
        logger.info("Adding session_id and formatting event_date")
        data = data.withColumn(
            "session_id",
            F.concat(F.col("user_pseudo_id"), F.col("event_params.ga_session_id")),
        )
        data = data.withColumn("event_date", F.to_date(F.col("event_date"), "yyyyMMdd"))
        return data.select(
            "user_pseudo_id", "user_id", "session_id", "event_name", "event_date",
            "event_timestamp", "batch_event_index", "geo", "device", "items",
            "user_properties", "event_params", *additional_columns_to_select,
        )

    def explode_items(self, data: DataFrame) -> DataFrame:
        """Explodes the 'items' column."""
        logger.info("Exploding 'items' column")
        return data.withColumn("items", F.explode("items"))

    def items_to_array(self, data: DataFrame) -> DataFrame:
        """Converts items column to array."""
        logger.info("Converting 'items' column to array")
        window = Window.partitionBy("user_pseudo_id", "session_id", "event_timestamp")
        data = data.withColumn("items", F.collect_list("items").over(window))
        return data.dropDuplicates(["user_pseudo_id", "session_id", "event_timestamp"])

    def post_process(self, data: DataFrame) -> DataFrame:
        """Drops unnecessary fields from 'items' and 'event_params'."""
        logger.info("Post-processing: Removing unused fields from 'event_params'")
        event_params_fields_to_drop = ["user_pseudo_id", "user_id"]
        data = data.withColumn(
            "event_params", F.col("event_params").dropFields(*event_params_fields_to_drop)
        )
        return data

## 4. Load data directly from BigQuery

Uses the Spark BigQuery connector. Set `SOURCE_QUERY` in the configuration cell to read a wildcard/date range instead of a single table (requires `viewsEnabled=true` and a materialization dataset).

In [ ]:
# readDataFormat=AVRO: the connector's default Arrow reader fails on GA4's deeply
# nested array<struct> columns (items, event_params, user_properties) with
# "should have as many children as in the schema: found 0 expected N". AVRO reads
# these nested schemas correctly.
if SOURCE_QUERY:
    logger.info("Loading data from BigQuery via query")
    data = (
        spark.read.format("bigquery")
        .option("query", SOURCE_QUERY)
        .option("viewsEnabled", "true")
        .option("materializationDataset", DESTINATION_DATASET)
        .option("readDataFormat", "AVRO")
        .load()
    )
else:
    logger.info(f"Loading data from BigQuery table: {SOURCE_TABLE}")
    data = (
        spark.read.format("bigquery")
        .option("table", SOURCE_TABLE)
        .option("readDataFormat", "AVRO")
        .load()
    )

logger.info("Data loaded. Starting job execution.")

## 5. Run the transformation

In [ ]:
job = GA4FlattenEventsJob(spark)

data_to_save = job.process_data(
    data,
    EVENT_NAME,
    EVENT_TYPE,
    ADDITIONAL_COLUMNS_TO_SELECT,
    EVENT_LIST,
    EVENT_PATTERN,
)

data_to_save.printSchema()

## 6. Write directly to BigQuery

Uses the `direct` write method (BigQuery Storage Write API) — no temporary GCS bucket required. The destination table is created partitioned by `event_date` (DAY) and clustered by `user_pseudo_id`, matching the original job.

**Partition-overwrite caveat:** the `direct` write method does **not** support partition decorators (`table$YYYYMMDD`), so `mode("overwrite")` replaces the **entire table**, not just the `RUN_DATE` partition. This is fine for a full backfill or a single-day placeholder run.

If you need to overwrite only one day's partition (the original scheduled behavior), switch to the **indirect** write method, which supports `datePartition` — at the cost of staging through a GCS bucket:

```python
(
    data_to_save.write.format("bigquery")
    .option("writeMethod", "indirect")
    .option("temporaryGcsBucket", "your-temp-bucket")
    .option("partitionField", PARTITION_FIELD)
    .option("partitionType", "DAY")
    .option("datePartition", RUN_DATE_NODASH)   # targets partition table$YYYYMMDD
    .option("clusteredFields", CLUSTER_FIELDS)
    .option("intermediateFormat", "orc")
    .mode("overwrite")
    .save(target)
)
```

In [ ]:
target = f"{DESTINATION_PROJECT}.{DESTINATION_DATASET}.{DESTINATION_TABLE}"

# DataFrame.isEmpty() -- RDD API is unavailable on Spark Connect (BigQuery managed Spark).
if data_to_save.isEmpty():
    logger.warning("No data to write. Skipping BigQuery write.")
else:
    logger.info(f"Data processed. Writing directly to BigQuery table: {target}")
    # writeMethod=direct (Storage Write API) does NOT support partition decorators,
    # so the original job's `datePartition` (table$YYYYMMDD) option is omitted -- it
    # produced an invalid table ID like "purchase$20240101". The table is still
    # created partitioned by event_date (DAY) and clustered by user_pseudo_id;
    # mode("overwrite") replaces the WHOLE table rather than a single day's partition.
    # See the note below to overwrite just one partition instead.
    (
        data_to_save.write.format("bigquery")
        .option("writeMethod", "direct")
        .option("partitionField", PARTITION_FIELD)
        .option("partitionType", "DAY")
        .option("clusteredFields", CLUSTER_FIELDS)
        .option("allowFieldAddition", "true")
        .option("allowFieldRelaxation", "true")
        .mode("overwrite")
        .save(target)
    )
    logger.info(f"Data saved to BigQuery table: {target}")